In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from conus_biomass import dir_info

In [2]:
dir_processed_model_output = dir_info.dir_model_output[:-1] + "_processed/"

In [3]:
for i in np.arange(0, 50):
    model_suffix = f"_{i:04d}"
    # print(model_suffix)
    df = pd.read_csv(dir_processed_model_output + "MMTC_our_study" + model_suffix + ".csv")
    df = df.drop(columns=["Unnamed: 0"])
    df["model_suffix"] = model_suffix
    if i == 0:
        df_all = df
    else:
        df_all = pd.concat([df_all, df])

In [4]:
df_all_westwide = df_all.groupby(["year", "model_suffix"]).sum()["live_biomass_MMT"].reset_index()

In [5]:
# Get the 2005 baseline for each model_suffix
baseline = df_all_westwide[df_all_westwide["year"] == 2005].set_index("model_suffix")[
    "live_biomass_MMT"
]

# Subtract the matching 2005 value based on model_suffix
df_all_westwide["live_biomass_MMT_delta"] = df_all_westwide["live_biomass_MMT"] - df_all_westwide[
    "model_suffix"
].map(baseline)

df_all_westwide["live_biomass_MMT_frac_delta"] = df_all_westwide[
    "live_biomass_MMT"
] / df_all_westwide["model_suffix"].map(baseline)

In [6]:
df_all_westwide.to_csv("OurStudy_western_stocks.csv")

In [7]:
# Get the 2005 baseline for each model_suffix
baseline = df_all_westwide[df_all_westwide["year"] == 2005].set_index("model_suffix")[
    "live_biomass_MMT"
]

# Subtract the matching 2005 value based on model_suffix
df_all_westwide["live_biomass_MMT_delta"] = df_all_westwide["live_biomass_MMT"] - df_all_westwide[
    "model_suffix"
].map(baseline)

df_all_westwide["live_biomass_MMT_frac_delta"] = df_all_westwide[
    "live_biomass_MMT"
] / df_all_westwide["model_suffix"].map(baseline)

In [8]:
non_CONUS = ["Alaska", "Hawaii", "U.S. Territories", "Puerto Rico"]
western_states = [
    "California",
    "Washington",
    "Oregon",
    "Idaho",
    "Montana",
    "Arizona",
    "Colorado",
    "New Mexico",
    "Utah",
    "Wyoming",
    "Nevada",
]
years = np.arange(2005, 2023)

FRF_state = pd.read_csv(dir_info.dir_walters + "FRF_stock_by_State.csv")

FRF_state_agb = FRF_state[FRF_state["Carbon Pools"] == "Aboveground Biomass"].drop(
    columns=["Carbon Pools"]
)
FRF_state_conus = FRF_state_agb[~FRF_state_agb["State"].isin(non_CONUS)]

CONUS_stocks = FRF_state_conus.drop(columns=["State"]).sum()
CONUS_stocks.index = CONUS_stocks.index.astype(int)

FRF_state_western = FRF_state_conus[FRF_state_conus["State"].isin(western_states)]
western_stocks = FRF_state_western.drop(columns=["State"]).sum()
western_stocks.index = western_stocks.index.astype(int)

FRF_state_eastern = FRF_state_conus[~FRF_state_conus["State"].isin(western_states)]
eastern_stocks = FRF_state_eastern.drop(columns=["State"]).sum()
eastern_stocks.index = eastern_stocks.index.astype(int)

western_stocks_delta = western_stocks - (western_stocks[western_stocks.index == years[0]].values)
eastern_stocks_delta = eastern_stocks - (eastern_stocks[eastern_stocks.index == years[0]].values)
CONUS_stocks_delta = CONUS_stocks - (CONUS_stocks[CONUS_stocks.index == years[0]].values)

USFS_years = CONUS_stocks.index

In [9]:
western_stocks.to_csv("USFS_western_stocks.csv")